# 🧩 0) Environment & Imports

In [1]:
# %% [markdown]
# # Sigma-only NeRF — Train / Resume / Evaluate
# This notebook:
# 1) loads a tiny scene,
# 2) builds SigmaMLP,
# 3) trains or resumes from a checkpoint,
# 4) (optionally) renders full opacity & sigma.
#
# Assumes your custom package layout:
#   nerflab.nerf_sigma_learning.{SigmaMLP, TrainCfg, train_sigma_only, ...}

# %%
from pathlib import Path
import os
import torch

# --- your custom libs (keep paths as in your code) ---
from nerflab.nerf_sigma_learning import (
    SigmaMLP,
    TrainCfg,
    train_sigma_only,
    normalize_optim_for_device,
    inspect_optim_state,   # optional, for debug
)
from nerflab.nerf_sigma_learning.learning_utils import (
    count_params,
    load_model_weights,
    make_warmup_cosine_lambda,
    load_checkpoint_full,
)

# nerflab core
from nerflab import Camera
from nerflab.io import load_batch_simple, get_frame_ids_for_case
from nerflab.nerf_sigma_learning.ops.forward_sigma import nerf_opacity

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.7.1+cu128
CUDA available: True


# ⚙️ 1) Config Cell

In [2]:
# %%
# ---------- user config ----------
scene_dir     = Path("../../data/data_experiment2")
split         = "train"
seed          = 7
num_frames    = 5
frame_offset  = 0

# Rays per pose per batch (tip: reduce on CPU to e.g. 1024–2048)
K             = 4096

# Device: let the trainer resolve "auto" → CUDA if available else CPU
device        = torch.device("cuda")

# ---------- model ----------
L_posenc      = 8
hidden_dim    = 64
depth         = 4
density_bias  = -1.0

# ---------- training ----------
steps         = 50_000
print_every   = 100
eval_every    = 100
lr            = 5e-4
weight_decay  = 1e-4
warmup        = 500
grad_clip     = 1.0
amp           = True                         # AMP is used only on CUDA
chunk_train   = 32_768
chunk_eval    = 65_536
eval_idx      = 0

# ---------- checkpoints ----------
ckpt_dir  = scene_dir
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_best = ckpt_dir / "sigma_best.pt"
ckpt_last = ckpt_dir / "sigma_last.pt"

# ---------- resume options ----------
# If you want to start from scratch: set resume = None
resume        = None                  # e.g., ckpt_best or ckpt_last; or None
reset_optim   = False                      # True = weights-only; False = full resume
extra_steps   = 0                     # Train for N additional steps (on top of resume step)
new_lr        = None                       # e.g., 2e-4 to override LR on resume
rewarm_steps  = None                       # e.g., 300 to rebuild warmup on resume

# Determinism knobs (optional)
# torch.use_deterministic_algorithms(True)
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [3]:
# %% [markdown]
# ## RNG (match trainer device)
# Build the generator on the resolved device to avoid cross-device RNG errors.

rng = torch.Generator(device=device).manual_seed(seed)


# 📦 2) Load Data (one-time)

In [4]:
# %% [markdown]
# ## Load batch (images, poses)

# %%
# The IO helpers accept a device; we pass a placeholder here.
# The trainer will also move/normalize devices internally.
frame_ids, _ = get_frame_ids_for_case(
    scene_dir, split, seed=seed, num_frames=num_frames, frame_offset=frame_offset
)

# Temporarily pick CUDA if available (the trainer will re-place tensors anyway)
batch = load_batch_simple(scene_dir, frame_ids, device=device)

images = batch["images"]          # (B, H, W) in [0,1]
H_wc   = batch["H_wc"]            # (B, 4, 4)
B, H, W = images.shape
images_flat = images.view(B, -1)

print(f"Loaded {B} frames at {H}x{W}")

Loaded 5 frames at 480x640


# 🔭 3) Ray Sampler (kept as your API)

In [5]:
# %% [markdown]
# ## Ray sampling helper (uses your nerflab Camera)

# %%
cam = Camera(H_wc)

def get_batch_rays(debug: bool = False):
    """
    Returns:
      C_true: (B, R)
      delta : (B, R, N)
      pts   : (B, R, N, 3)
    """
    Osub, Dsub, idx_lin, uv_int = cam.get_rays_sampled(
        rays_per_pose=K, rng=rng, return_indices=True
    )
    t, delta, pts = cam.sample_along_rays(Osub, Dsub, rng=rng)
    C_true = torch.gather(images_flat, 1, idx_lin)

    if debug:
        print("subset shapes: O", Osub.shape, "D", Dsub.shape)
        print("sample shapes:", "t", t.shape, "delta", delta.shape, "pts", pts.shape)
        print("idx_lin:", idx_lin.shape, "| uv_int:", uv_int.shape)

    return C_true, delta, pts

# quick sanity (optional)
_ = get_batch_rays(debug=True)

subset shapes: O torch.Size([5, 4096, 3]) D torch.Size([5, 4096, 3])
sample shapes: t torch.Size([5, 4096, 40]) delta torch.Size([5, 4096, 40]) pts torch.Size([5, 4096, 40, 3])
idx_lin: torch.Size([5, 4096]) | uv_int: torch.Size([5, 4096, 2])


# 🧠 4) Build Model

In [6]:
# %% [markdown]
# ## Model

# %%
model = SigmaMLP(
    L_posenc=L_posenc,
    hidden_dim=hidden_dim,
    depth=depth,
    skip_at=(2,),
    density_bias=density_bias,
)
print(f"SigmaMLP params: {count_params(model):,}")

SigmaMLP params: 14,977


# 🏋️ 5) Trainer Config

In [7]:
# %% [markdown]
# ## Train config (TrainCfg)

# %%
cfg = TrainCfg(
    steps=steps,
    lr=lr,
    weight_decay=weight_decay,
    warmup_steps=warmup,
    grad_clip_norm=grad_clip,
    amp=amp,
    print_every=print_every,
    eval_every=eval_every,
    device=device,                  # ← let the trainer resolve/own the device
    chunk_rays_train=chunk_train,
    chunk_rays_eval=chunk_eval,
    ckpt_best_path=str(ckpt_best),
    ckpt_last_path=str(ckpt_last),
    eval_idx=eval_idx,
)
cfg


TrainCfg(steps=50000, lr=0.0005, weight_decay=0.0001, warmup_steps=500, grad_clip_norm=1.0, amp=True, amp_dtype='auto', compile_mode=None, grad_accum_steps=1, print_every=100, eval_every=100, device=device(type='cuda'), chunk_rays_train=32768, chunk_rays_eval=65536, ckpt_best_path='../../data/data_experiment2/sigma_best.pt', ckpt_last_path='../../data/data_experiment2/sigma_last.pt', eval_idx=0)

# 🔁 6) Resume Logic (weights-only or full)

In [8]:
# %% [markdown]
# ## Resume logic
# - reset_optim=False → full resume (model + optimizer + scheduler + scaler)
# - reset_optim=True  → load weights only (fresh optimizer/scheduler)
# - extra_steps: extends cfg.steps to (resume_step + extra_steps)
# - new_lr, rewarm_steps: optional overrides on resume
#
# Notes:
# • Ensure model is on the target device before loading weights.
# • Normalize optimizer state tensors (Adam moments & 'step') to device.
# • Use GradScaler(device="cuda", enabled=...) for modern AMP APIs.

# %%
from typing import Dict, Any

injected_optim   = None
injected_sched   = None
injected_scaler  = None
start_step       = 0

# Place the model on an initial device (trainer will normalize too)
model.to(device)

if (resume is not None) and Path(resume).exists():
    if reset_optim:
        # --- Weights-only resume: load model weights; train fresh optimizer/scheduler ---
        _ = load_model_weights(model, str(resume), device)
        if extra_steps is not None:
            # Fresh run for exactly `extra_steps`
            cfg.steps = int(extra_steps)
        start_step = 0
        print("Resume mode: weights-only (fresh optimizer/scheduler).")
    else:
        # --- Full resume: rebuild optim/sched/scaler, then load full checkpoint ---
        optim = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr,
            betas=(0.9, 0.99),
            weight_decay=cfg.weight_decay,
            foreach=False,       # avoid mixed-device foreach quirks
            capturable=(device.type == "cuda"),     # keep internal 'step' buffer on CUDA
        )
        sched = torch.optim.lr_scheduler.LambdaLR(
            optim, lr_lambda=make_warmup_cosine_lambda(cfg.steps, cfg.warmup_steps)
        )
        scaler = torch.amp.GradScaler(device="cuda", enabled=(cfg.amp and device.type == "cuda"))

        # Load the full checkpoint (model + optim + sched + scaler + step, etc.)
        ckpt: Dict[str, Any] = load_checkpoint_full(str(resume), model, optim, sched, scaler, device)
        ckpt_step = int(ckpt.get("step", 0))
        start_step = ckpt_step

        # Migrate any CPU optimizer state (Adam moments, 'step') to device
        normalize_optim_for_device(optim, device, enable_capturable_on_cuda=True)
        # Optional: print mismatches if any
        # inspect_optim_state(optim)

        # Optional overrides
        if new_lr is not None:
            for g in optim.param_groups:
                g["lr"] = float(new_lr)

        if rewarm_steps is not None:
            sched = torch.optim.lr_scheduler.LambdaLR(
                optim, lr_lambda=make_warmup_cosine_lambda(cfg.steps, int(rewarm_steps))
            )

        # Extend total steps: resume_step + extra_steps
        if extra_steps is not None:
            cfg.steps = ckpt_step + int(extra_steps)

        # Hand off injected objects to the trainer
        injected_optim  = optim
        injected_sched  = sched
        injected_scaler = scaler
else:
    print("No checkpoint to resume from. Starting fresh.")

print(
    f"Resume from: {resume if (resume and Path(resume).exists()) else 'None'} | "
    f"start_step={start_step} | total target steps={cfg.steps}"
)

No checkpoint to resume from. Starting fresh.
Resume from: None | start_step=0 | total target steps=50000


# 🚀 7) Train

In [ ]:
# %% [markdown]
# ## Train / Continue training

# %%
_ = train_sigma_only(
    model=model,
    get_batch_rays=get_batch_rays,
    H_wc=H_wc,
    images=images,
    rng=rng,
    cfg=cfg,
    optim=injected_optim,
    sched=injected_sched,
    scaler=injected_scaler,
    start_step=start_step,
)

[     1/50000] loss=0.140428 | PSNR= 8.53 dB | lr=2.00e-06 | rays=20,480 | 0.2s | max_mem=1.38 GB
[   100/50000] loss=0.131345 | PSNR= 8.82 dB | lr=1.01e-04 | rays=20,480 | 5.6s | max_mem=1.40 GB
  → eval frame 0: MSE=0.125380, PSNR=9.02 dB
  ✓ saved BEST checkpoint: ../../data/data_experiment2/sigma_best.pt (PSNR=9.02 dB)
[   200/50000] loss=0.099267 | PSNR=10.03 dB | lr=2.01e-04 | rays=20,480 | 11.5s | max_mem=3.40 GB
  → eval frame 0: MSE=0.082554, PSNR=10.83 dB
  ✓ saved BEST checkpoint: ../../data/data_experiment2/sigma_best.pt (PSNR=10.83 dB)
[   300/50000] loss=0.062937 | PSNR=12.01 dB | lr=3.01e-04 | rays=20,480 | 17.3s | max_mem=3.40 GB
  → eval frame 0: MSE=0.034547, PSNR=14.62 dB
  ✓ saved BEST checkpoint: ../../data/data_experiment2/sigma_best.pt (PSNR=14.62 dB)
[   400/50000] loss=0.043451 | PSNR=13.62 dB | lr=4.01e-04 | rays=20,480 | 23.2s | max_mem=3.40 GB
  → eval frame 0: MSE=0.022685, PSNR=16.44 dB
  ✓ saved BEST checkpoint: ../../data/data_experiment2/sigma_best.pt (

: 